In [2]:
!pip install rdkit PyTDC scikit-learn xgboost pandas matplotlib -q

In [3]:
from tdc.multi_pred import DTI
data = DTI(name='DAVIS')
df = data.get_data()
print(df.shape)
df.head()

Found local copy...
Loading...
Done!


(25772, 5)


,Drug_ID,Drug,Target_ID,Target,Y
0,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,AAK1,MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...,43.0
1,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ABL1p,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0
2,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ABL2,MVLGTVLLPPNSYGRDQDTSLCCLCTEASESALPDLTDHFASCVED...,10000.0
3,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ACVR1,MVDGVMILPVLIMIALPSPSMEDEKPKVNPKLYMCVCEGLSCGNED...,10000.0
4,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ACVR1B,MAESAGASSFFPLVVLLLAGSGGSGPRGVQALLCACTSCLQANYTC...,10000.0


In [4]:
split = data.get_split(method='cold_split', column_name='Drug')
train = split['train']
valid = split['valid']
test = split['test']

print("Train size:", train.shape)
print("Valid size:", valid.shape)
print("Test size:", test.shape)

Train size: (17813, 5)
Valid size: (2653, 5)
Test size: (5306, 5)


In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_fp(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    return np.array(fp)

# Test it on one molecule first
sample_smiles = train['Drug'].iloc[0]
fp = smiles_to_fp(sample_smiles)
print("SMILES:", sample_smiles)
print("Fingerprint shape:", fp.shape)
print("Fingerprint (first 20 values):", fp[:20])

SMILES: CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5CCOCC5)ccc34)cc2)no1
Fingerprint shape: (2048,)
Fingerprint (first 20 values): [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]


In [6]:
def add_fingerprints(df):
    df = df.copy()
    df['fp'] = df['Drug'].apply(smiles_to_fp)
    return df

train_fp = add_fingerprints(train)
valid_fp = add_fingerprints(valid)
test_fp = add_fingerprints(test)

print("Train fingerprints done:", train_fp['fp'].notnull().sum(), "/", len(train_fp))

Train fingerprints done: 17813 / 17813


In [7]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

def protein_to_features(seq):
    seq = seq.upper()
    length = len(seq)
    if length == 0:
        return np.zeros(len(AMINO_ACIDS))
    counts = np.array([seq.count(aa) for aa in AMINO_ACIDS])
    return counts / length  # normalize to percentages

# Test on one protein
sample_protein = train['Target'].iloc[0]
pf = protein_to_features(sample_protein)
print("Protein length:", len(sample_protein))
print("Feature vector shape:", pf.shape)
print("Feature vector:", pf)

Protein length: 961
Feature vector shape: (20,)
Feature vector: [0.08220604 0.01352758 0.04682622 0.04474506 0.03433923 0.06763788
 0.01560874 0.04162331 0.05098855 0.08740895 0.01352758 0.03433923
 0.09781478 0.11654527 0.03329865 0.08324662 0.06451613 0.05306972
 0.00312175 0.01560874]


In [8]:
def add_protein_features(df):
    df = df.copy()
    df['protein_feat'] = df['Target'].apply(protein_to_features)
    return df

train_fp = add_protein_features(train_fp)
valid_fp = add_protein_features(valid_fp)
test_fp = add_protein_features(test_fp)

print("Done. Sample columns:", train_fp.columns.tolist())

Done. Sample columns: ['Drug_ID', 'Drug', 'Target_ID', 'Target', 'Y', 'fp', 'protein_feat']


In [9]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Combine fingerprint + protein features into one input matrix
def build_X(df):
    fp_matrix = np.stack(df['fp'].values)
    protein_matrix = np.stack(df['protein_feat'].values)
    return np.hstack([fp_matrix, protein_matrix])

X_train = build_X(train_fp)
X_valid = build_X(valid_fp)
X_test = build_X(test_fp)

y_train = train_fp['Y'].values
y_valid = valid_fp['Y'].values
y_test = test_fp['Y'].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (17813, 2068)
y_train shape: (17813,)


In [10]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

# Evaluate on test set
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {mse:.4f}")
print(f"Test R²: {r2:.4f}")

Test MSE: 16711973.0528
Test R²: -0.0038


In [11]:
print(train_fp['Y'].describe())
print(train_fp['Y'].min(), train_fp['Y'].max())

count    17813.000000
mean      7519.683947
std       4027.557743
min          0.016000
25%       3500.000000
50%      10000.000000
75%      10000.000000
max      10000.000000
Name: Y, dtype: float64
0.016 10000.0


In [12]:
train_fp['Y_log'] = np.log10(train_fp['Y'])
valid_fp['Y_log'] = np.log10(valid_fp['Y'])
test_fp['Y_log'] = np.log10(test_fp['Y'])

print(train_fp['Y_log'].describe())

count    17813.000000
mean         3.569726
std          0.869245
min         -1.795880
25%          3.544068
50%          4.000000
75%          4.000000
max          4.000000
Name: Y_log, dtype: float64


In [13]:
y_train = train_fp['Y_log'].values
y_valid = valid_fp['Y_log'].values
y_test = test_fp['Y_log'].values

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE (log scale): {mse:.4f}")
print(f"Test R²: {r2:.4f}")

Test MSE (log scale): 0.6472
Test R²: -0.0413


In [14]:
print("TRAIN Y_log:")
print(train_fp['Y_log'].describe())
print()
print("TEST Y_log:")
print(test_fp['Y_log'].describe())

TRAIN Y_log:
count    17813.000000
mean         3.569726
std          0.869245
min         -1.795880
25%          3.544068
50%          4.000000
75%          4.000000
max          4.000000
Name: Y_log, dtype: float64

TEST Y_log:
count    5306.000000
mean        3.582025
std         0.788439
min        -1.000000
25%         3.431364
50%         4.000000
75%         4.000000
max         4.000000
Name: Y_log, dtype: float64


In [15]:
print("Actual y_test range:", y_test.min(), "to", y_test.max())
print("Predicted y_pred range:", y_pred.min(), "to", y_pred.max())
print("Predicted mean:", y_pred.mean(), "  Actual mean:", y_test.mean())
print("Predicted std:", y_pred.std(), "  Actual std:", y_test.std())

Actual y_test range: -1.0 to 4.0
Predicted y_pred range: 0.25193208 to 4.2319236
Predicted mean: 3.7363586   Actual mean: 3.5820245931251065
Predicted std: 0.33854842   Actual std: 0.7883650347096982


In [16]:
y_train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
print("Train R²:", train_r2)

Train R²: 0.7248882382258467


In [17]:
model3 = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=3,             # much shallower trees
    learning_rate=0.03,
    subsample=0.8,           # each tree sees only 80% of rows
    colsample_bytree=0.5,    # each tree sees only 50% of features
    reg_alpha=1.0,           # L1 regularization
    reg_lambda=2.0,          # L2 regularization
    random_state=42,
    early_stopping_rounds=30
)

model3.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

y_pred3 = model3.predict(X_test)
y_train_pred3 = model3.predict(X_train)

print("Train R²:", r2_score(y_train, y_train_pred3))
print("Test R²:", r2_score(y_test, y_pred3))
print("Test MSE:", mean_squared_error(y_test, y_pred3))
print("Best iteration:", model3.best_iteration)

Train R²: 0.23660915636078406
Test R²: -0.03864671801349395
Test MSE: 0.6455391140248246
Best iteration: 49


In [18]:
from sklearn.model_selection import train_test_split

# Quick random-split sanity check (temporary, not your real evaluation)
X_all = np.vstack([X_train, X_valid, X_test])
y_all = np.concatenate([y_train, y_valid, y_test])

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

sanity_model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
sanity_model.fit(X_tr, y_tr)
y_sanity_pred = sanity_model.predict(X_te)

print("RANDOM SPLIT Test R²:", r2_score(y_te, y_sanity_pred))

RANDOM SPLIT Test R²: 0.4198535730676686


In [19]:
import json

results = {
    "baseline_random_split": {
        "model": "XGBoost (Morgan FP + amino acid composition)",
        "test_r2": float(r2_score(y_te, y_sanity_pred)),
        "note": "Similar molecules can appear in both train/test — optimistic estimate"
    },
    "baseline_scaffold_split": {
        "model": "XGBoost (Morgan FP + amino acid composition)",
        "test_r2": float(r2_score(y_test, y_pred3)),
        "test_mse": float(mean_squared_error(y_test, y_pred3)),
        "note": "Realistic evaluation — test molecules are structurally novel vs train"
    }
}

with open("baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

{
  "baseline_random_split": {
    "model": "XGBoost (Morgan FP + amino acid composition)",
    "test_r2": 0.4198535730676686,
    "note": "Similar molecules can appear in both train/test \u2014 optimistic estimate"
  },
  "baseline_scaffold_split": {
    "model": "XGBoost (Morgan FP + amino acid composition)",
    "test_r2": -0.03864671801349395,
    "test_mse": 0.6455391140248246,
    "note": "Realistic evaluation \u2014 test molecules are structurally novel vs train"
  }
}


In [20]:
# ---------- CELL 1: Install libraries ----------
!pip install PyTDC --no-deps -q
!pip install fuzzywuzzy -q
!pip install rdkit xgboost -q


# ---------- CELL 2: Imports ----------
import numpy as np
import pandas as pd
import json
from itertools import product
from rdkit import Chem
from rdkit.Chem import AllChem
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tdc.multi_pred import DTI


# ---------- CELL 3: Load dataset ----------
data = DTI(name='DAVIS')
df = data.get_data()
print("Full dataset shape:", df.shape)


# ---------- CELL 4: Scaffold split (train/valid/test) ----------
split = data.get_split(method='cold_split', column_name='Drug')
train = split['train'].copy()
valid = split['valid'].copy()
test = split['test'].copy()
print("Train:", train.shape, "| Valid:", valid.shape, "| Test:", test.shape)


# ---------- CELL 5: Molecule featurization (Morgan fingerprints) ----------
def smiles_to_fp(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    return np.array(fp)

def add_fingerprints(d):
    d = d.copy()
    d['fp'] = d['Drug'].apply(smiles_to_fp)
    before = len(d)
    d = d[d['fp'].notnull()].reset_index(drop=True)
    dropped = before - len(d)
    if dropped > 0:
        print(f"  Dropped {dropped} rows with unparseable SMILES")
    return d

train = add_fingerprints(train)
valid = add_fingerprints(valid)
test = add_fingerprints(test)
print("Fingerprints done. Train:", train.shape, "Valid:", valid.shape, "Test:", test.shape)


# ---------- CELL 6: Protein featurization (mono + dipeptide composition) ----------
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
DIPEPTIDES = [''.join(p) for p in product(AMINO_ACIDS, repeat=2)]

def protein_to_features(seq):
    seq = str(seq).upper()
    length = len(seq)
    if length < 2:
        return np.zeros(len(AMINO_ACIDS) + len(DIPEPTIDES))
    mono = np.array([seq.count(aa) for aa in AMINO_ACIDS]) / length
    di_counts = {}
    for i in range(length - 1):
        pair = seq[i:i+2]
        di_counts[pair] = di_counts.get(pair, 0) + 1
    di = np.array([di_counts.get(dp, 0) for dp in DIPEPTIDES]) / (length - 1)
    return np.concatenate([mono, di])

def add_protein_features(d):
    d = d.copy()
    d['protein_feat'] = d['Target'].apply(protein_to_features)
    return d

train = add_protein_features(train)
valid = add_protein_features(valid)
test = add_protein_features(test)
print("Protein features done. Feature length:", len(train['protein_feat'].iloc[0]))


# ---------- CELL 7: Log-transform target (raw Kd -> pKd-like scale) ----------
train['Y_log'] = np.log10(train['Y'])
valid['Y_log'] = np.log10(valid['Y'])
test['Y_log'] = np.log10(test['Y'])
print("Y_log range - train:", train['Y_log'].min(), "to", train['Y_log'].max())


# ---------- CELL 8: Build final feature matrices ----------
def build_X(d):
    fp_matrix = np.stack(d['fp'].values)
    protein_matrix = np.stack(d['protein_feat'].values)
    return np.hstack([fp_matrix, protein_matrix])

X_train = build_X(train)
X_valid = build_X(valid)
X_test = build_X(test)

y_train = train['Y_log'].values
y_valid = valid['Y_log'].values
y_test = test['Y_log'].values

print("X_train shape:", X_train.shape)


# ---------- CELL 9: Train baseline model (regularized, scaffold split) ----------
model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.5,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    early_stopping_rounds=30
)
model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

y_pred = model.predict(X_test)
scaffold_r2 = r2_score(y_test, y_pred)
scaffold_mse = mean_squared_error(y_test, y_pred)
print(f"SCAFFOLD SPLIT -> Test R²: {scaffold_r2:.4f} | Test MSE: {scaffold_mse:.4f}")


# ---------- CELL 10: Random-split sanity check (control experiment) ----------
X_all = np.vstack([X_train, X_valid, X_test])
y_all = np.concatenate([y_train, y_valid, y_test])
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

sanity_model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
sanity_model.fit(X_tr, y_tr)
y_sanity_pred = sanity_model.predict(X_te)
random_r2 = r2_score(y_te, y_sanity_pred)
print(f"RANDOM SPLIT (control) -> Test R²: {random_r2:.4f}")


# ---------- CELL 11: Save results ----------
results = {
    "baseline_random_split": {
        "model": "XGBoost (Morgan FP + mono+dipeptide protein composition)",
        "test_r2": float(random_r2),
        "note": "Similar molecules can appear in both train/test - optimistic estimate"
    },
    "baseline_scaffold_split": {
        "model": "XGBoost (Morgan FP + mono+dipeptide protein composition), regularized",
        "test_r2": float(scaffold_r2),
        "test_mse": float(scaffold_mse),
        "note": "Realistic evaluation - test molecules are structurally novel vs train"
    }
}
with open("baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

Found local copy...
Loading...
Done!


Full dataset shape: (25772, 5)
Train: (17813, 5) | Valid: (2653, 5) | Test: (5306, 5)
Fingerprints done. Train: (17813, 6) Valid: (2653, 6) Test: (5306, 6)
Protein features done. Feature length: 420
Y_log range - train: -1.7958800173440752 to 4.0
X_train shape: (17813, 2468)
SCAFFOLD SPLIT -> Test R²: -0.0015 | Test MSE: 0.6225
RANDOM SPLIT (control) -> Test R²: 0.4845
{
  "baseline_random_split": {
    "model": "XGBoost (Morgan FP + mono+dipeptide protein composition)",
    "test_r2": 0.4844504874554525,
    "note": "Similar molecules can appear in both train/test - optimistic estimate"
  },
  "baseline_scaffold_split": {
    "model": "XGBoost (Morgan FP + mono+dipeptide protein composition), regularized",
    "test_r2": -0.0015231907568586944,
    "test_mse": 0.6224661206006895,
    "note": "Realistic evaluation - test molecules are structurally novel vs train"
  }
}
